# Training Model Sentimen — Collection `comments_sentiment`

Notebook ini melatih model klasifikasi sentimen (**Logistic Regression** dan **Naive Bayes**) menggunakan **PySpark MLlib**.  
Data diambil dari MongoDB collection **`comments_sentiment`**, menggunakan field **`text_final`** sebagai fitur dan **`sentiment`** sebagai label target.

## Hasil Analisa Collection (pra-notebook)
| Informasi | Nilai |
|-----------|-------|
| Total dokumen | 2.000 |
| Label `negatif` | 1.009 (50,4%) |
| Label `netral` | 694 (34,7%) |
| Label `positif` | 297 (14,9%) |
| Field fitur | `text_final` (sudah preprocessed) |
| Field target | `sentiment` |

> **Catatan:** Data **imbalanced** — kelas `positif` sangat minoritas (14,9%).  
> Strategi mitigasi:
> - Split stratifikasi **80% train / 20% test** agar proporsi label terjaga di kedua set
> - `class_weight` pada Logistic Regression untuk kompensasi imbalance
> - Evaluasi lengkap: F1 weighted, F1 macro, precision & recall per kelas, confusion matrix
> - Semua hasil evaluasi disimpan ke MongoDB collection `sentiment_training_eval`

## 1. Import Library

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Iterator

from pymongo import MongoClient
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer,
    IDF,
    NGram,
    RegexTokenizer,
    StringIndexer,
    VectorAssembler,
)
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print('Library berhasil diimpor.', flush=True)

## 2. Setup Path & Import Modul Lokal

In [ ]:
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from mongo_comments_loader import (
        create_spark_session as create_project_spark_session,
        load_project_env,
    )
    print('Modul lokal (mongo_comments_loader) berhasil diimpor.')
except ImportError as exc:
    print(f'Error mengimpor modul lokal: {exc}')
    print('Pastikan notebook dijalankan dari root direktori proyek.')

## 3. Konstanta & Konfigurasi

In [ ]:
SEED         = 42
TRAIN_RATIO  = 0.80        # 80% train, 20% test
TEXT_COL     = 'text_final'  # Field fitur teks (sudah preprocessed)
LABEL_COL    = 'sentiment'   # Field target: positif / netral / negatif
VALID_LABELS = ('positif', 'netral', 'negatif')

MONGO_SOURCE_COLLECTION = 'comments_sentiment'
MONGO_EVAL_COLLECTION   = 'sentiment_training_eval'

print('Konfigurasi:')
print(f'  TEXT_COL          = {TEXT_COL}')
print(f'  LABEL_COL         = {LABEL_COL}')
print(f'  VALID_LABELS      = {VALID_LABELS}')
print(f'  Train/Test ratio  = {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')
print(f'  Source collection = {MONGO_SOURCE_COLLECTION}')
print(f'  Eval collection   = {MONGO_EVAL_COLLECTION}')

## 4. Load Environment & Inisialisasi Spark

In [ ]:
load_project_env()

MONGO_URI = os.getenv('MONGO_URI', '').strip()
MONGO_DB  = os.getenv('MONGO_DB', 'analisis_sentimen').strip()

if not MONGO_URI:
    raise ValueError('MONGO_URI belum diisi di .env')
if not MONGO_DB:
    raise ValueError('MONGO_DB belum diisi di .env')

print(f'MongoDB DB  : {MONGO_DB}')
print(f'Source col  : {MONGO_SOURCE_COLLECTION}')

spark = create_project_spark_session(
    app_name='sentiment-training-comments-sentiment',
    cores=int(os.getenv('SPARK_CORES', '4')),
    memory=os.getenv('SPARK_MEMORY', '2g'),
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark aktif: master={spark.sparkContext.master}')

## 5. Load & Eksplorasi Data dari MongoDB

In [ ]:
def _normalize_value(val):
    """Konversi dict/list ke JSON string agar kompatibel dengan Spark."""
    if isinstance(val, (dict, list)):
        return json.dumps(val, ensure_ascii=False, default=str)
    return val


def load_data(spark: SparkSession) -> DataFrame:
    """Load dokumen dari comments_sentiment, filter hanya text_final + sentiment valid."""
    docs = []
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        cursor = client[MONGO_DB][MONGO_SOURCE_COLLECTION].find(
            {
                TEXT_COL:  {'$exists': True, '$ne': ''},
                LABEL_COL: {'$exists': True, '$nin': [None, '']},
            },
            {'_id': 0, TEXT_COL: 1, LABEL_COL: 1}
        )
        for doc in cursor:
            docs.append({k: _normalize_value(v) for k, v in doc.items()})

    if not docs:
        raise ValueError(f'Collection {MONGO_DB}.{MONGO_SOURCE_COLLECTION} kosong atau tidak valid.')

    df = spark.createDataFrame(docs)
    df = (
        df.select(
            F.col(TEXT_COL).cast('string').alias(TEXT_COL),
            F.col(LABEL_COL).cast('string').alias(LABEL_COL),
        )
        .filter(F.col(TEXT_COL).isNotNull() & (F.trim(F.col(TEXT_COL)) != ''))
        .filter(F.col(LABEL_COL).isin(*VALID_LABELS))
    )
    return df


# Load data
raw_df = load_data(spark).cache()
total = raw_df.count()
print(f'Total data valid dimuat: {total}')

### 5.1 Eksplorasi Distribusi Label & Statistik Teks

In [ ]:
print('=== Distribusi Label ===')
label_dist = raw_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL)
label_dist.show(truncate=False)

# Persentase
print('Persentase label:')
for row in label_dist.collect():
    print(f'  {row[LABEL_COL]:10s}: {row["count"]:5d} ({row["count"]/total*100:.1f}%)')

print()
print('=== Statistik Panjang Teks (karakter) ===')
raw_df.select(
    F.length(F.col(TEXT_COL)).alias('char_len')
).describe().show()

print('=== Statistik Jumlah Token (kata) ===')
raw_df.select(
    F.size(F.split(F.col(TEXT_COL), r'\s+')).alias('token_count')
).describe().show()

print('=== Sample 5 Data ===')
raw_df.show(5, truncate=80)

## 6. Stratified Split 80% Train / 20% Test

In [ ]:
def stratified_split_80_20(df: DataFrame, seed: int = SEED):
    """
    Split stratifikasi presisi: menjaga proporsi label di train dan test.
    Menggunakan row_number per label, lalu 80% pertama -> train, sisanya -> test.
    """
    window_spec = Window.partitionBy(LABEL_COL).orderBy(F.rand(seed))
    df_ranked = df.withColumn('_rn', F.row_number().over(window_spec))

    label_counts = {r[LABEL_COL]: r['count'] for r in df.groupBy(LABEL_COL).count().collect()}

    train_cond = None
    test_cond  = None
    for lbl in VALID_LABELS:
        cnt = label_counts.get(lbl, 0)
        if cnt == 0:
            continue
        train_limit = int(round(cnt * TRAIN_RATIO))
        c_train = (F.col(LABEL_COL) == lbl) & (F.col('_rn') <= train_limit)
        c_test  = (F.col(LABEL_COL) == lbl) & (F.col('_rn') >  train_limit)
        train_cond = c_train if train_cond is None else train_cond | c_train
        test_cond  = c_test  if test_cond  is None else test_cond  | c_test

    train_df = df_ranked.filter(train_cond).drop('_rn').cache()
    test_df  = df_ranked.filter(test_cond).drop('_rn').cache()
    return train_df, test_df


train_df, test_df = stratified_split_80_20(raw_df)

train_count = train_df.count()
test_count  = test_df.count()
print(f'Train: {train_count} ({train_count/total*100:.1f}%)')
print(f'Test : {test_count}  ({test_count/total*100:.1f}%)')

print()
print('Distribusi label TRAIN:')
train_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show(truncate=False)

print('Distribusi label TEST:')
test_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show(truncate=False)

## 7. Definisi Pipeline ML (TF-IDF + Classifier)

In [ ]:
def build_pipeline(model_name: str) -> Pipeline:
    """
    Bangun pipeline:
      RegexTokenizer -> NGram(2) -> CountVectorizer unigram+bigram
      -> [IDF jika LR] -> VectorAssembler -> StringIndexer -> Classifier

    Data imbalanced: LR menggunakan weightCol (class weight) dan elasticNet.
    NB: multinomial dengan smoothing, tidak menggunakan IDF (butuh nilai non-negatif).
    """
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol='tokens',
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol='label_index', handleInvalid='skip'
    )
    ngram = NGram(n=2, inputCol='tokens', outputCol='bigrams')

    name = model_name.lower().strip()

    if name in ('logistic_regression', 'lr', 'logistic regression'):
        cv_uni = CountVectorizer(
            inputCol='tokens', outputCol='uni_feat', vocabSize=8000, minDF=2.0
        )
        cv_bi = CountVectorizer(
            inputCol='bigrams', outputCol='bi_feat', vocabSize=6000, minDF=2.0
        )
        assembler = VectorAssembler(
            inputCols=['uni_feat', 'bi_feat'], outputCol='raw_feat'
        )
        idf = IDF(inputCol='raw_feat', outputCol='features', minDocFreq=2)
        clf = LogisticRegression(
            featuresCol='features', labelCol='label_index',
            predictionCol='pred_index',
            maxIter=200, regParam=0.05, elasticNetParam=0.15,
            family='multinomial',
            # weightCol diset secara manual setelah fit — lihat train_and_evaluate
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]

    elif name in ('naive_bayes', 'nb', 'naive bayes'):
        cv_uni = CountVectorizer(
            inputCol='tokens', outputCol='uni_feat', vocabSize=8000, minDF=2.0
        )
        cv_bi = CountVectorizer(
            inputCol='bigrams', outputCol='bi_feat', vocabSize=6000, minDF=2.0
        )
        assembler = VectorAssembler(
            inputCols=['uni_feat', 'bi_feat'], outputCol='features'
        )
        clf = NaiveBayes(
            featuresCol='features', labelCol='label_index',
            predictionCol='pred_index',
            modelType='multinomial', smoothing=1.0,
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, label_indexer, clf]

    else:
        raise ValueError(f'Model tidak dikenal: {model_name}')

    return Pipeline(stages=stages)


print('Fungsi build_pipeline siap digunakan.')

## 8. Class Weight untuk Data Imbalanced (Logistic Regression)

In [ ]:
def add_class_weights(df: DataFrame, label_index_col: str = 'label_index') -> DataFrame:
    """
    Tambahkan kolom 'class_weight' menggunakan inverse frequency weighting.
    Formula: weight(c) = total / (n_classes * count(c))
    Ini membantu model memperhatikan kelas minoritas (positif) lebih.
    """
    total = df.count()
    n_classes = len(VALID_LABELS)
    counts = {r[label_index_col]: r['cnt'] for r in
              df.groupBy(label_index_col).agg(F.count('*').alias('cnt')).collect()}

    weight_map = {idx: total / (n_classes * cnt) for idx, cnt in counts.items()}
    print('Class weights (inverse frequency):')
    for idx, w in sorted(weight_map.items()):
        print(f'  label_index={idx:.0f} -> weight={w:.4f}')

    def get_weight(idx):
        return float(weight_map.get(idx, 1.0))

    weight_udf = F.udf(get_weight, 'double')
    return df.withColumn('class_weight', weight_udf(F.col(label_index_col)))


print('Fungsi add_class_weights siap digunakan.')

## 9. Fungsi Evaluasi Lengkap

In [ ]:
def compute_metrics(predictions: DataFrame, labels: list) -> dict:
    """
    Hitung metrik evaluasi lengkap:
      - Accuracy
      - F1 weighted & macro
      - Precision weighted & macro
      - Recall weighted & macro
      - Per-class: precision, recall, f1, support
      - Confusion matrix (sebagai dict)
    """
    total = predictions.count()

    def _eval(metric):
        return MulticlassClassificationEvaluator(
            labelCol='label_index', predictionCol='pred_index', metricName=metric
        ).evaluate(predictions)

    accuracy          = _eval('accuracy')
    f1_weighted       = _eval('f1')
    precision_weighted = _eval('weightedPrecision')
    recall_weighted    = _eval('weightedRecall')

    # Confusion matrix dari groupBy actual vs predicted label
    cm_rows = {
        (r[LABEL_COL], r['pred_label']): r['cnt']
        for r in predictions.groupBy(LABEL_COL, 'pred_label')
                             .agg(F.count('*').alias('cnt'))
                             .collect()
    }

    # Per-class metrics
    per_class = {}
    macro_p = macro_r = macro_f1 = 0.0
    for lbl in VALID_LABELS:
        tp = cm_rows.get((lbl, lbl), 0)
        fp = sum(cm_rows.get((actual, lbl), 0) for actual in VALID_LABELS if actual != lbl)
        fn = sum(cm_rows.get((lbl, pred),   0) for pred   in VALID_LABELS if pred   != lbl)
        support = tp + fn
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r  = tp / support    if support   > 0 else 0.0
        f1 = 2*p*r / (p+r)  if (p+r)     > 0 else 0.0
        per_class[lbl] = {'precision': p, 'recall': r, 'f1': f1, 'support': support}
        macro_p  += p
        macro_r  += r
        macro_f1 += f1

    n = len(VALID_LABELS)
    macro_p  /= n
    macro_r  /= n
    macro_f1 /= n

    # Confusion matrix sebagai nested dict
    cm_dict = {
        actual: {pred: cm_rows.get((actual, pred), 0) for pred in VALID_LABELS}
        for actual in VALID_LABELS
    }

    return {
        'total_samples':      total,
        'accuracy':           round(accuracy, 6),
        'f1_weighted':        round(f1_weighted, 6),
        'f1_macro':           round(macro_f1, 6),
        'precision_weighted': round(precision_weighted, 6),
        'precision_macro':    round(macro_p, 6),
        'recall_weighted':    round(recall_weighted, 6),
        'recall_macro':       round(macro_r, 6),
        'per_class':          {k: {m: round(v, 6) if isinstance(v, float) else v
                                   for m, v in cls.items()}
                               for k, cls in per_class.items()},
        'confusion_matrix':   cm_dict,
    }


def print_metrics(metrics: dict, model_name: str, split: str) -> None:
    """Tampilkan ringkasan metrik secara terformat."""
    print(f'\n{"="*60}')
    print(f'  {model_name} | {split}')
    print(f'{"="*60}')
    print(f'  Total sampel      : {metrics["total_samples"]}')
    print(f'  Accuracy          : {metrics["accuracy"]:.4f}')
    print(f'  F1 Weighted       : {metrics["f1_weighted"]:.4f}')
    print(f'  F1 Macro          : {metrics["f1_macro"]:.4f}')
    print(f'  Precision Weighted: {metrics["precision_weighted"]:.4f}')
    print(f'  Precision Macro   : {metrics["precision_macro"]:.4f}')
    print(f'  Recall Weighted   : {metrics["recall_weighted"]:.4f}')
    print(f'  Recall Macro      : {metrics["recall_macro"]:.4f}')
    print()
    print(f'  {"Label":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
    print(f'  {"-"*56}')
    for lbl, m in metrics['per_class'].items():
        print(f'  {lbl:<12} {m["precision"]:>10.4f} {m["recall"]:>10.4f} {m["f1"]:>10.4f} {m["support"]:>10}')
    print()
    print('  Confusion Matrix (actual \\ predicted):')
    header = f'  {"":12}' + ''.join(f'{p:>12}' for p in VALID_LABELS)
    print(header)
    for actual in VALID_LABELS:
        row = f'  {actual:<12}' + ''.join(f'{metrics["confusion_matrix"][actual].get(p, 0):>12}' for p in VALID_LABELS)
        print(row)


print('Fungsi evaluasi siap.')

## 10. Fungsi Simpan Hasil Evaluasi ke MongoDB

In [ ]:
def save_eval_to_mongo(record: dict) -> None:
    """Simpan satu record hasil evaluasi ke collection sentiment_training_eval."""
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        client[MONGO_DB][MONGO_EVAL_COLLECTION].insert_one(record)
    print(f'  Evaluasi disimpan: {MONGO_DB}.{MONGO_EVAL_COLLECTION}')


print('Fungsi penyimpanan MongoDB siap.')

## 11. Fungsi Train & Evaluasi Pipeline

In [ ]:
def train_and_evaluate(
    model_key: str,
    model_display: str,
    train_df: DataFrame,
    test_df: DataFrame,
) -> dict:
    """
    1. Fit pipeline pada train_df
    2. Untuk LR: tambahkan class_weight setelah label_indexer fit
    3. Transform train & test
    4. Hitung metrik lengkap
    5. Simpan ke MongoDB
    Mengembalikan dict berisi metrik train dan test.
    """
    print(f'\n>>> Training: {model_display} ...', flush=True)

    pipeline = build_pipeline(model_key)
    is_lr = model_key.lower() in ('logistic_regression', 'lr', 'logistic regression')

    if is_lr:
        # Fit label indexer saja dulu untuk mendapat label mapping
        # Kemudian fit full pipeline dengan class_weight
        from pyspark.ml.feature import StringIndexer as SI
        from pyspark.ml.classification import LogisticRegression as LR

        # Partial fit: hanya StringIndexer + feature stages untuk mendapat label_index
        # Caranya: fit pipeline biasa, lalu ambil label_indexer model untuk tahu mapping
        fitted = pipeline.fit(train_df)
        # label_indexer ada di stages[-2] untuk LR (sebelum classifier)
        li_model = fitted.stages[-2]
        labels = list(li_model.labels)

        # Transform train untuk tambah class_weight
        # Gunakan sub-pipeline (semua kecuali classifier) untuk mendapat label_index
        from pyspark.ml import PipelineModel
        feature_stages = fitted.stages[:-1]  # semua kecuali LR
        feature_model = PipelineModel(stages=feature_stages)
        train_feat = feature_model.transform(train_df)
        train_weighted = add_class_weights(train_feat, 'label_index')

        # Re-fit LR saja dengan class_weight
        lr_stage = pipeline.getStages()[-1]
        lr_stage.setWeightCol('class_weight')
        pipeline_stages = pipeline.getStages()
        # Rebuild pipeline dengan LR yang sudah ada weightCol
        new_pipeline = Pipeline(stages=pipeline_stages)
        fitted = new_pipeline.fit(train_weighted)
        li_model = fitted.stages[-2]
        labels = list(li_model.labels)
    else:
        fitted = pipeline.fit(train_df)
        li_model = fitted.stages[-2]
        labels = list(li_model.labels)

    print(f'  Label mapping: {dict(enumerate(labels))}', flush=True)

    # UDF mapping pred_index -> pred_label
    def map_pred(idx):
        if idx is None: return None
        i = int(idx)
        return labels[i] if 0 <= i < len(labels) else None
    map_udf = F.udf(map_pred, 'string')

    # Transform dan evaluasi
    results = {}
    for split_name, split_df in [('train', train_df), ('test', test_df)]:
        preds = fitted.transform(split_df if split_name == 'test' else (
            train_weighted if is_lr else train_df
        ))
        preds = preds.withColumn('pred_label', map_udf(F.col('pred_index'))).cache()
        metrics = compute_metrics(preds, labels)
        print_metrics(metrics, model_display, split_name)

        # Simpan ke MongoDB
        record = {
            'model_name':       model_display,
            'model_key':        model_key,
            'split':            split_name,
            'source_collection': MONGO_SOURCE_COLLECTION,
            'text_col':         TEXT_COL,
            'label_col':        LABEL_COL,
            'train_size':       train_df.count() if split_name == 'train' else None,
            'test_size':        test_df.count()  if split_name == 'test'  else None,
            'label_mapping':    dict(enumerate(labels)),
            'trained_at':       datetime.now(timezone.utc).isoformat(),
            **metrics,
        }
        # confusion_matrix dan per_class sudah berupa dict, aman untuk MongoDB
        save_eval_to_mongo(record)
        results[split_name] = metrics
        preds.unpersist()

    return results


print('Fungsi train_and_evaluate siap.')

## 12. Jalankan Training & Evaluasi Semua Model

In [ ]:
all_results = {}

MODELS = [
    ('logistic_regression', 'Logistic Regression'),
    ('naive_bayes',         'Naive Bayes'),
]

try:
    for model_key, model_display in MODELS:
        results = train_and_evaluate(
            model_key=model_key,
            model_display=model_display,
            train_df=train_df,
            test_df=test_df,
        )
        all_results[model_display] = results

except Exception as exc:
    print(f'Error selama training: {exc}', flush=True)
    raise

print('\n>>> Semua model selesai dilatih dan dievaluasi.')

## 13. Ringkasan Perbandingan Model

In [ ]:
print('=== Ringkasan Hasil Evaluasi (TEST SET) ===')
print(f'{"Model":<22} {"Accuracy":>10} {"F1-W":>10} {"F1-M":>10} {"Prec-W":>10} {"Rec-W":>10}')
print('-' * 76)
for name, res in all_results.items():
    m = res['test']
    print(
        f'{name:<22}'
        f' {m["accuracy"]:>10.4f}'
        f' {m["f1_weighted"]:>10.4f}'
        f' {m["f1_macro"]:>10.4f}'
        f' {m["precision_weighted"]:>10.4f}'
        f' {m["recall_weighted"]:>10.4f}'
    )
print()
print('F1-W = F1 Weighted  |  F1-M = F1 Macro  |  Prec-W = Precision Weighted  |  Rec-W = Recall Weighted')
print()
print('=== Per-Kelas F1 (TEST SET) ===')
print(f'{"Model":<22} {"positif":>12} {"netral":>12} {"negatif":>12}')
print('-' * 62)
for name, res in all_results.items():
    pc = res['test']['per_class']
    print(
        f'{name:<22}'
        f' {pc.get("positif", {}).get("f1", 0):>12.4f}'
        f' {pc.get("netral",  {}).get("f1", 0):>12.4f}'
        f' {pc.get("negatif", {}).get("f1", 0):>12.4f}'
    )

## 14. Stop Spark Session

In [ ]:
spark.stop()
print('Spark session dihentikan.')
print(f'Hasil evaluasi tersimpan di MongoDB: {MONGO_DB}.{MONGO_EVAL_COLLECTION}')